# 05 — Calibration Alignment

**Purpose:** Use a new Neon lamp frame to measure small detector shifts relative to the existing CMOS wavelength table, fit a rigid correction, and optionally write a session settings file plus adjusted wavelength lookup table.

**When to run:** After a suspected small optics or detector shift. This is lighter than rebuilding the full wavelength calibration from scratch.

**Inputs:**
- Existing `pattern_CMOS_20240305.txt` and `Th_wavelength_CMOS_20240305.txt`
- New local Neon lamp `.sif` and matching background `.sif`
- New local sphere and sphere background `.sif` for order extraction / absolute-calibration context

**Outputs, if enabled:**
- `alignments/lhd_cmos_alignment_20250926.settings.toml`
- `alignments/Th_wavelength_CMOS_20240305_aligned_to_20250926.txt`

The historical calibration table is not overwritten.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import echelle_spectra
from echelle_spectra.tools.echelle import Calibrations, EchelleImage
from echelle_spectra.tools.calibration_alignment import (
    AlignmentSettings,
    apply_rigid_correction_to_lines,
    detector_points_from_lines,
    fit_rigid_transform,
    load_wavelength_table,
    measure_detector_window_saturation,
    measure_line_centroids,
    measure_line_window_stats,
    rank_candidate_lines,
    save_alignment_settings,
    select_candidate_lines,
    write_wavelength_table,
)

%matplotlib inline


## Configuration

The notebook defaults to the local folder created for the 2025-09-26 calibration pass. It discovers background files with either `-bg` or `_bg` suffixes.

In [ ]:
BASE_PATH = echelle_spectra._config["base_path"]
REPO_ROOT = BASE_PATH.parents[1]
CALIB_DIR = BASE_PATH / "resources" / "calibration_files"
LOCAL_CALIB_DIR = REPO_ROOT / "local" / "20250926_calib"
ALIGNMENT_DIR = CALIB_DIR / "alignments"

BASE_WAVELENGTH_FILE = "Th_wavelength_CMOS_20240305.txt"
BASE_PATTERN_FILE = "pattern_CMOS_20240305.txt"
ALIGNMENT_DATASET_ID = "20250926"
ALIGNMENT_SOURCE_LABEL = "local/20250926_calib"
ALIGNMENT_LAMP = "Ne"
CREATED_AT = "2026-06-04"

# Prefer the short Neon exposure first. It is bright enough for lines, but the GUI shows no clipping.
NEON_FILE_HINT = "Ne-0.02s-x3-bright-lines"

# First-pass 16-bit full-scale guard. Use raw 2D detector pixels for this, not extracted spectra.
SATURATION_LEVEL = 0.98 * 65535
MIN_SNR = 5.0
WINDOW_RADIUS_PX = 18
N_PREVIEW_LINES = 12

SAVE_OUTPUTS = False
SETTINGS_OUT = ALIGNMENT_DIR / f"lhd_cmos_alignment_{ALIGNMENT_DATASET_ID}.settings.toml"
ADJUSTED_TABLE_NAME = f"Th_wavelength_CMOS_20240305_aligned_to_{ALIGNMENT_DATASET_ID}.txt"
ADJUSTED_TABLE_OUT = ALIGNMENT_DIR / ADJUSTED_TABLE_NAME

print(f"CALIB_DIR:       {CALIB_DIR}")
print(f"LOCAL_CALIB_DIR: {LOCAL_CALIB_DIR}")
print(f"ALIGNMENT_DIR:   {ALIGNMENT_DIR}")


## Discover local SIF files

In [ ]:
def is_background(path: Path) -> bool:
    stem = path.stem.lower()
    return stem.endswith("-bg") or stem.endswith("_bg")


def without_background_suffix(path: Path) -> str:
    stem = path.stem
    lower = stem.lower()
    if lower.endswith("-bg") or lower.endswith("_bg"):
        return stem[:-3]
    return stem


def find_signal_and_background(folder: Path, contains: str) -> tuple[Path, Path]:
    files = sorted(folder.glob("*.sif"))
    matches = [p for p in files if contains.lower() in p.stem.lower()]
    signal = [p for p in matches if not is_background(p)]
    background = [p for p in matches if is_background(p)]
    if not signal:
        raise FileNotFoundError(f"No signal SIF containing {contains!r} in {folder}")
    sig = signal[0]
    sig_key = sig.stem.lower()
    bg_candidates = [p for p in background if without_background_suffix(p).lower() == sig_key]
    if not bg_candidates:
        raise FileNotFoundError(f"No matching background for {sig.name}")
    return sig, bg_candidates[0]


neon_path, neon_bg_path = find_signal_and_background(LOCAL_CALIB_DIR, NEON_FILE_HINT)
sphere_path, sphere_bg_path = find_signal_and_background(LOCAL_CALIB_DIR, "sphere")

for label, path in [
    ("Neon", neon_path),
    ("Neon background", neon_bg_path),
    ("Sphere", sphere_path),
    ("Sphere background", sphere_bg_path),
]:
    print(f"{label:18s}: {path.name}")

## Load calibration context and Neon frames

`Calibrations` still uses one folder plus filenames. Absolute local SIF paths are accepted by `os.path.join` on Windows, so we can keep base text files in `resources/calibration_files` and use local SIF files without copying them.

In [ ]:
files_cmos = {
    "orders": BASE_PATTERN_FILE,
    "wavelength": BASE_WAVELENGTH_FILE,
    "sphr": str(sphere_path),
    "bkgr": str(sphere_bg_path),
    "integral": "integrating_sphere.txt",
}

cb = Calibrations(folder=str(CALIB_DIR), filenames=files_cmos)
cb.start()
print("Calibration context loaded.")
print(f"Detector: {cb.DIMO} rows x {cb.DIMW} cols")
print(f"Orders:   {cb.pattern.shape[1]}")

In [ ]:
neon = EchelleImage(str(neon_path), clbr=cb)
neon.calculate_order_spectra()
neon.correct_order_shapes()

neon_bg = EchelleImage(str(neon_bg_path), clbr=cb)
neon_bg.calculate_order_spectra()
neon_bg.correct_order_shapes()

raw_order_spectra = np.asarray(neon.order_spectra[0], dtype=float)
raw_background_spectra = np.asarray(neon_bg.order_spectra[0], dtype=float)
order_spectra = raw_order_spectra - raw_background_spectra
print(f"Neon order spectra: {order_spectra.shape}")
print(f"Raw peak count:      {np.nanmax(raw_order_spectra):.1f}")
print(f"Saturation guard:    {SATURATION_LEVEL:.1f}")


## Select and rank curated Neon candidate lines


In [ ]:
rows = load_wavelength_table(CALIB_DIR / BASE_WAVELENGTH_FILE)
candidates = select_candidate_lines(
    rows,
    species=("NeI",),
    require_ok=True,
    min_width_px=4,
    max_width_px=40,
)

candidate_df = pd.DataFrame([line.__dict__ for line in candidates])
print(f"Active rows: {len(rows)}")
print(f"Candidate NeI rows: {len(candidates)}")
candidate_df.head(12)


## Diagnose candidate windows before fitting

Saturation is estimated from raw 2D detector pixels in a small box around each expected line position. The extracted 1D order spectrum is an integrated trace, so it can be much larger than 65535 even when the detector itself is not clipped. The background-subtracted 1D spectrum is used for SNR ranking and Gaussian fitting because it better represents local line prominence.


In [ ]:
def line_key(line):
    return (line.order_idx, line.center_pixel, line.wavelength_nm)

signal_ranked_stats = rank_candidate_lines(
    order_spectra,
    candidates,
    window_radius_px=WINDOW_RADIUS_PX,
    saturation_level=None,
    min_snr=MIN_SNR,
)
detector_saturation = measure_detector_window_saturation(
    neon.images,
    cb.pattern,
    candidates,
    x_radius_px=WINDOW_RADIUS_PX,
    y_radius_px=4,
    saturation_level=SATURATION_LEVEL,
)
saturation_by_key = {line_key(stat.line): stat for stat in detector_saturation}

ranked_stats = []
for stat in signal_ranked_stats:
    sat_stat = saturation_by_key[line_key(stat.line)]
    if sat_stat.is_saturated:
        ranked_stats.append((stat, sat_stat, False, "saturated"))
    else:
        ranked_stats.append((stat, sat_stat, stat.fit_candidate, stat.reason))

window_df = pd.DataFrame([
    {
        "order": stat.line.order_idx,
        "species": stat.line.species,
        "wavelength_nm": stat.line.wavelength_nm,
        "expected_px": stat.line.center_pixel,
        "peak_px": stat.peak_pixel,
        "peak_dx_px": stat.peak_pixel - stat.line.center_pixel,
        "sub_peak": stat.peak_value,
        "sub_prominence": stat.prominence,
        "sub_snr": stat.snr,
        "detector_peak": sat_stat.peak_value,
        "detector_saturated_px": sat_stat.saturated_pixels,
        "detector_saturated_fraction": sat_stat.saturated_fraction,
        "fit_candidate": fit_candidate,
        "reason": reason,
        "comment": stat.line.comment,
    }
    for stat, sat_stat, fit_candidate, reason in ranked_stats
])

print("Candidate-window reasons:")
print(window_df.groupby(["fit_candidate", "reason"], dropna=False).size())
window_df.head(N_PREVIEW_LINES)


In [ ]:
def plot_line_windows(stats_with_saturation, spectra, max_lines=12, title="Candidate windows"):
    picked = stats_with_saturation[:max_lines]
    if not picked:
        print("No line windows to plot.")
        return

    ncols = 3
    nrows = int(np.ceil(len(picked) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 2.8 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, (stat, sat_stat, fit_candidate, reason) in zip(axes_flat, picked):
        line = stat.line
        x0 = int(round(line.center_pixel))
        lo = max(0, x0 - WINDOW_RADIUS_PX)
        hi = min(spectra.shape[1], x0 + WINDOW_RADIUS_PX + 1)
        x = np.arange(lo, hi)
        y = spectra[line.order_idx, lo:hi]

        ax.plot(x, y, color="tab:blue", lw=1.4, label="bg-sub")
        ax.axvline(line.center_pixel, color="0.2", ls="--", lw=1, label="expected")
        ax.axvline(stat.peak_pixel, color="tab:orange", ls=":", lw=1.2, label="1D peak")
        if sat_stat.is_saturated:
            ax.text(0.02, 0.92, "2D clipped", transform=ax.transAxes, color="tab:red")

        status = "fit" if fit_candidate else reason
        ax.set_title(f"o{line.order_idx} {line.wavelength_nm:.3f} nm | {status}")
        ax.set_xlabel("pixel")
        ax.set_ylabel("bg-sub counts")

    for ax in axes_flat[len(picked):]:
        ax.axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

plot_line_windows(ranked_stats, order_spectra, N_PREVIEW_LINES, "Best ranked candidate windows")


## Fit line centroids

In [ ]:
fit_candidates = [stat.line for stat, sat_stat, fit_candidate, reason in ranked_stats]
fits = measure_line_centroids(
    order_spectra,
    fit_candidates,
    window_radius_px=WINDOW_RADIUS_PX,
    saturation_level=None,
    min_snr=MIN_SNR,
)

fit_df = pd.DataFrame([
    {
        "order": fit.line.order_idx,
        "species": fit.line.species,
        "wavelength_nm": fit.line.wavelength_nm,
        "expected_px": fit.line.center_pixel,
        "measured_px": fit.center_pixel,
        "dx_px": fit.center_pixel - fit.line.center_pixel,
        "sigma_px": fit.sigma_px,
        "amplitude": fit.amplitude,
        "snr": fit.snr,
        "pre_snr": fit.diagnostics.snr if fit.diagnostics else np.nan,
        "raw_peak": saturation_by_key[line_key(fit.line)].peak_value,
        "detector_saturated_px": saturation_by_key[line_key(fit.line)].saturated_pixels,
        "success": fit.success,
        "reason": "saturated" if saturation_by_key[line_key(fit.line)].is_saturated else fit.reason,
    }
    for fit in fits
])

# Keep detector-saturated windows visible in the report, but do not trust their fits downstream.
good = [fit for fit in fits if fit.success and not saturation_by_key[line_key(fit.line)].is_saturated]
print(f"Successful non-saturated fits: {len(good)} / {len(fits)}")
print("Fit reasons:")
print(fit_df.groupby(["success", "reason"], dropna=False).size())
fit_df.sort_values(["success", "detector_saturated_px", "snr"], ascending=[True, False, False]).head(20)


In [ ]:
failed_keys = {(row.order, row.expected_px, row.wavelength_nm) for row in fit_df[~fit_df["success"]].itertuples()}
failed_ranked = [
    item for item in ranked_stats
    if (item[0].line.order_idx, item[0].line.center_pixel, item[0].line.wavelength_nm) in failed_keys
]
plot_line_windows(failed_ranked, order_spectra, min(N_PREVIEW_LINES, len(failed_ranked)), "Representative failed windows")


## Inspect centroid shifts

In [ ]:
good_df = fit_df[fit_df["success"] & (fit_df["detector_saturated_px"] == 0)].copy()

if good_df.empty:
    print("No successful non-saturated fits yet. Inspect the ranked and failed line windows above.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].axhline(0, color="0.6", lw=1)
    axes[0].scatter(good_df["expected_px"], good_df["dx_px"], c=good_df["order"], cmap="viridis")
    axes[0].set_xlabel("Expected center pixel")
    axes[0].set_ylabel("Measured - expected (px)")
    axes[0].set_title("Centroid shift by x")

    axes[1].axhline(0, color="0.6", lw=1)
    axes[1].scatter(good_df["order"], good_df["dx_px"], c=good_df["wavelength_nm"], cmap="plasma")
    axes[1].set_xlabel("Order index")
    axes[1].set_ylabel("Measured - expected (px)")
    axes[1].set_title("Centroid shift by order")

    plt.tight_layout()
    plt.show()

    display(good_df[["order", "wavelength_nm", "expected_px", "measured_px", "dx_px", "snr", "sigma_px"]].describe())


## Fit rigid detector correction

In [ ]:
transform = None
adjusted_rows = None
rms_px = np.nan
residual_px = np.array([])
residual_xy = np.empty((0, 2))
expected_xy = np.empty((0, 2))
measured_xy = np.empty((0, 2))
predicted_xy = np.empty((0, 2))
good_lines = []

if len(good) < 2:
    print("Need at least two successful non-saturated line fits for a rigid transform.")
else:
    good_lines = [fit.line for fit in good]
    measured_centers = [fit.center_pixel for fit in good]

    expected_xy = detector_points_from_lines(good_lines, cb.pattern)
    measured_xy = detector_points_from_lines(good_lines, cb.pattern, measured_centers)

    transform, rms_px = fit_rigid_transform(expected_xy, measured_xy)
    predicted_xy = transform.apply(expected_xy)
    residual_xy = predicted_xy - measured_xy
    residual_px = np.sqrt(np.sum(residual_xy**2, axis=1))

    print("Rigid transform expected -> measured")
    print(f"  dx       = {transform.dx_px:.4f} px")
    print(f"  dy       = {transform.dy_px:.4f} px")
    print(f"  theta    = {transform.theta_deg:.5f} deg")
    print(f"  RMS      = {rms_px:.4f} px")
    print(f"  n lines  = {len(good_lines)}")


In [ ]:
if transform is None:
    print("Rigid-fit plots skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].scatter(measured_xy[:, 0], measured_xy[:, 1], s=20, label="measured")
    axes[0].scatter(predicted_xy[:, 0], predicted_xy[:, 1], s=14, label="rigid fit")
    axes[0].invert_yaxis()
    axes[0].set_xlabel("x pixel")
    axes[0].set_ylabel("y pixel")
    axes[0].set_title("Detector points")
    axes[0].legend()

    axes[1].hist(residual_px, bins=20, color="tab:blue", alpha=0.8)
    axes[1].set_xlabel("Residual (px)")
    axes[1].set_ylabel("Lines")
    axes[1].set_title("Rigid-fit residuals")

    plt.tight_layout()
    plt.show()

    display(pd.DataFrame({
        "order": [line.order_idx for line in good_lines],
        "wavelength_nm": [line.wavelength_nm for line in good_lines],
        "residual_px": residual_px,
        "residual_x_px": residual_xy[:, 0],
        "residual_y_px": residual_xy[:, 1],
    }).sort_values("residual_px", ascending=False).head(12))


## Preview adjusted wavelength table

In [ ]:
if transform is None:
    print("Adjusted wavelength preview skipped until a rigid transform is available.")
else:
    adjusted_rows = apply_rigid_correction_to_lines(rows, cb.pattern, transform)
    preview = pd.DataFrame([
        {
            "order": new.order_idx,
            "species": new.species,
            "wavelength_nm": new.wavelength_nm,
            "old_center_px": old.center_pixel,
            "new_center_px": new.center_pixel,
            "dx_px": new.center_pixel - old.center_pixel,
            "comment": new.comment,
        }
        for old, new in zip(rows, adjusted_rows)
    ])
    display(preview.head(20))


## Save settings and adjusted lookup table

Leave `SAVE_OUTPUTS = False` until the residuals look physically reasonable. Saving writes new files; it never overwrites the historical lookup table.

In [ ]:
if transform is None or adjusted_rows is None:
    print("No transform available; nothing to save.")
else:
    settings = AlignmentSettings(
        instrument_id="lhd_cmos",
        created_at=CREATED_AT,
        alignment_dataset_id=ALIGNMENT_DATASET_ID,
        alignment_source_dir=ALIGNMENT_SOURCE_LABEL,
        alignment_lamp=ALIGNMENT_LAMP,
        signal_file=neon_path.name,
        background_file=neon_bg_path.name,
        base_wavelength_file=BASE_WAVELENGTH_FILE,
        base_pattern_file=BASE_PATTERN_FILE,
        sphere_file=sphere_path.name,
        sphere_background_file=sphere_bg_path.name,
        output_wavelength_file=ADJUSTED_TABLE_NAME,
        transform=transform,
        n_lines=len(good_lines),
        rms_px=rms_px,
        notes="Rigid detector correction from Neon lamp; pattern not regenerated in this notebook.",
    )
    table_metadata = [
        ("Generated", CREATED_AT),
        ("Base wavelength file", BASE_WAVELENGTH_FILE),
        ("Base pattern file", BASE_PATTERN_FILE),
        ("Alignment dataset", ALIGNMENT_DATASET_ID),
        ("Alignment source dir", ALIGNMENT_SOURCE_LABEL),
        ("Signal", neon_path.name),
        ("Background", neon_bg_path.name),
        ("Sphere", sphere_path.name),
        ("Sphere background", sphere_bg_path.name),
        ("Correction model", "rigid detector transform, dx/dy/theta"),
        ("Settings file", SETTINGS_OUT.name),
        ("Note", "pattern not regenerated in this notebook"),
    ]

    if SAVE_OUTPUTS:
        ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)
        save_alignment_settings(settings, SETTINGS_OUT)
        write_wavelength_table(adjusted_rows, ADJUSTED_TABLE_OUT, metadata=table_metadata)
        print(f"Saved settings: {SETTINGS_OUT}")
        print(f"Saved adjusted table: {ADJUSTED_TABLE_OUT}")
    else:
        print("SAVE_OUTPUTS=False - not writing files.")
        print(f"Settings would be: {SETTINGS_OUT}")
        print(f"Adjusted table would be: {ADJUSTED_TABLE_OUT}")


## Next validation

After the Neon residuals look good, point a normal extraction workflow at `alignments/Th_wavelength_CMOS_20240305_aligned_to_20250926.txt` and check real LHD emission:

1. Rough check: H-alpha, H-beta, H-gamma.
2. Precise check: identified Fulcher lines.
3. Record systematic offsets in nm and compare with the previous suspected ~0.1 nm error.
